# Summit-Sim: Complete E2E Workflow Demo

Demonstrates the full Summit-Sim workflow:
1. **Scenario Generation** - AI generates wilderness rescue scenario
2. **Simulation** - Student navigates scenario with human-in-the-loop
3. **Debrief** - Performance analysis and learning report

All phases are traced to MLflow for visibility.

## 1. Setup & Imports

In [20]:
import warnings

# Suppress autologging warnings
warnings.filterwarnings("ignore", message=".*autologging.*")
warnings.filterwarnings("ignore", message=".*LangChain.*")

from summit_sim.agents.generator import generate_scenario
from summit_sim.graphs.simulation import create_simulation_graph
from summit_sim.schemas import HostConfig, generate_scenario_id
from summit_sim.settings import settings
from summit_sim.tracing import enable_tracing, simulation_session

# Initialize MLflow tracing
enable_tracing()

print(f"✓ MLflow: {settings.mlflow_tracking_uri}")
print(f"✓ Experiment: {settings.mlflow_experiment_name}")
print(f"✓ Model: {settings.default_model}")
print(f"✓ API Key: {bool(settings.openrouter_api_key)}")

✓ MLflow: https://mlflow.bhamm-lab.com
✓ Experiment: summit-sim
✓ Model: google/gemini-3.1-flash-lite-preview
✓ API Key: True


## 2. Phase 1: Scenario Generation

Generate a complete wilderness rescue scenario from minimal host configuration.

In [21]:
# Configuration
config = HostConfig(
    num_participants=3,
    activity_type="hiking",
    difficulty="med",
    class_id="demo-class-2024",
)

print(
    f"Generating: {config.activity_type} scenario for {config.num_participants} participants"
)
print("-" * 60)

# Generate scenario
scenario = await generate_scenario(config)
scenario_id = generate_scenario_id()

print(f"✓ Title: {scenario.title}")
print(f"✓ Setting: {scenario.setting}")
print(f"✓ Patient: {scenario.patient_summary}")
print(f"✓ Turns: {len(scenario.turns)}")
print(f"✓ Scenario ID: {scenario_id}")
print("\nLearning Objectives:")
for obj in scenario.learning_objectives:
    print(f"  • {obj}")

Generating: hiking scenario for 3 participants
------------------------------------------------------------
✓ Title: The Silent Breath: Alpine Emergency
✓ Setting: Alpine hiking trail at 8,000ft elevation. Late afternoon, temperature dropping rapidly (approx 45°F / 7°C). The team is 4 miles from the trailhead.
✓ Patient: 30-year-old female, Sarah. Fit, regular hiker. History: No known asthma or heart conditions. Rapidly ascended to 8,000 ft in 8 hours. Currently experiencing extreme lethargy, rapid shallow breathing, dry cough, and mild bluish hue to lips.
✓ Turns: 4
✓ Scenario ID: scn-c84cf23d

Learning Objectives:
  • Correctly identify symptoms of HAPE and differentiate them from asthma or simple exhaustion.
  • Prioritize descent as the definitive field treatment for acute altitude illness.
  • Understand the risks of patient exertion in HAPE patients and the importance of passive transport.


Trace(trace_id=tr-a2235dd0fce0dbbcaab4560c5c32e3a8)

### Display Scenario Structure

In [22]:
for turn in scenario.turns:
    marker = "(START)" if turn.is_starting_turn else ""
    print(f"\n{'=' * 60}")
    print(f"Turn {turn.turn_id} {marker}")
    print(f"{'=' * 60}")
    print(
        turn.narrative_text[:200] + "..."
        if len(turn.narrative_text) > 200
        else turn.narrative_text
    )
    print("\nChoices:")
    for choice in turn.choices:
        mark = "✓" if choice.is_correct else " "
        next_turn = (
            "END" if choice.next_turn_id is None else f"Turn {choice.next_turn_id}"
        )
        print(f"  [{mark}] {choice.description[:60]}... → {next_turn}")


Turn 0 (START)
You are hiking just above 8,000ft when Sarah stops abruptly. She is breathless, pale, and leaning on her trekking poles. Her resting heart rate appears very high, and she has developed a dry, persiste...

Choices:
  [ ] Assume it is severe exhaustion and altitude fatigue. Encoura... → Turn 2
  [✓] Suspect altitude-related pulmonary issue. Immediately initia... → Turn 1

Turn 1 
You have correctly identified that staying at this altitude is dangerous. Sarah is now struggling to walk and her breathing has become more laboured with a slight rattle detectable in her chest. What ...

Choices:
  [✓] Prioritize organized, slow descent. Assist Sarah by carrying... → Turn 3
  [ ] In a rush to get her to lower elevation, assist her in walki... → Turn 3

Turn 2 
After 30 minutes of rest, Sarah hasn't improved. In fact, her lips are clearly cyanotic (blue), and she is now coughing up thin, frothy, pink-tinged sputum. She can barely stand up. You realize the de...

Choices:
  [✓] Re

## 3. Phase 2: Simulation

Run the interactive simulation with human-in-the-loop.
The graph will pause at each turn for you to make a choice.

In [23]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# Create graph with memory
memory = InMemorySaver()
graph = create_simulation_graph(checkpointer=memory)

# Initialize state
initial_state = {
    "scenario_draft": scenario,
    "current_turn_id": scenario.starting_turn_id,
    "transcript": [],
    "is_complete": False,
    "key_learning_moments": [],
    "last_selected_choice": None,
    "simulation_result": None,
    "scenario_id": scenario_id,
    "class_id": config.class_id,
    "debrief_report": None,
}

print("✓ Graph initialized")
print(f"✓ Starting at turn {initial_state['current_turn_id']}")

✓ Graph initialized
✓ Starting at turn 0


In [24]:
# Run simulation with MLflow session tracking
import mlflow

from summit_sim.tracing import log_simulation_results

with simulation_session(config, scenario_id=scenario_id) as (session_id, graph_config):
    print(f"\n📋 Session ID: {session_id}")
    print("🎮 Starting simulation...\n")
    print("=" * 60)
    print("INTERACTIVE SIMULATION")
    print("=" * 60 + "\n")

    # Link traces to session using metadata
    mlflow.update_current_trace(
        metadata={
            "mlflow.trace.session": session_id,
            "mlflow.trace.user": "student",
            "scenario_id": scenario_id,
        }
    )

    current_state = initial_state
    turn_count = 0
    simulation_complete = False

    print("\n=== STARTING SIMULATION ===\n")

    while not simulation_complete:
        input_state = initial_state if turn_count == 0 else None
        turn_count += 1
        print(f"--- TURN {turn_count} ---")

        async for event in graph.astream(input_state, graph_config):
            if "__interrupt__" in event:
                interrupt_obj = event["__interrupt__"]
                if hasattr(interrupt_obj, "value"):
                    interrupt_data = interrupt_obj.value
                elif isinstance(interrupt_obj, (list, tuple)):
                    interrupt_data = (
                        interrupt_obj[0].value
                        if hasattr(interrupt_obj[0], "value")
                        else interrupt_obj[0]
                    )
                else:
                    interrupt_data = interrupt_obj

                print(f"\n📖 {interrupt_data['narrative']}\n")
                print("Choices:")
                for i, choice in enumerate(interrupt_data["choices"], 1):
                    print(f"  {i}. {choice['description']}")

                while True:
                    try:
                        sel = int(input("\nSelect: ")) - 1
                        if 0 <= sel < len(interrupt_data["choices"]):
                            break
                        print("Invalid")
                    except ValueError:
                        print("Enter number")

                selected = interrupt_data["choices"][sel]
                current_state = await graph.ainvoke(
                    Command(resume={"choice_id": selected["choice_id"]}), graph_config
                )
                break

            # Check for graph completion
            if event == {}:
                simulation_complete = True
                break

        # Check if state indicates completion
        if current_state.get("is_complete"):
            simulation_complete = True

        # Safety limit
        if turn_count > 10:
            print("Safety stop")
            break

    print("\n=== SIMULATION COMPLETE ===")

    # Log final results to MLflow parent run
    log_simulation_results(
        transcript=current_state["transcript"],
        is_complete=current_state["is_complete"],
        key_learning_moments=current_state["key_learning_moments"],
        debrief_report=current_state["debrief_report"],
    )
    print("\n✓ Results logged to MLflow")
    print(f"✓ Total turns: {len(current_state['transcript'])}")
    print(f"✓ Learning moments: {len(current_state['key_learning_moments'])}")

2026/03/24 08:26:57 WARNING mlflow.tracing.fluent: No active trace found. Please create a span using `mlflow.start_span` or `@mlflow.trace` before calling `mlflow.update_current_trace`.



📋 Session ID: 5fd2c57e-91cd-4d85-b13f-21f88e3dd704
🎮 Starting simulation...

INTERACTIVE SIMULATION


=== STARTING SIMULATION ===

--- TURN 1 ---


2026/03/24 08:26:57 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f9db7adb4c0> at 0x7f9d798aa500> was created in a different Context
2026/03/24 08:26:57 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f9db7adb4c0> at 0x7f9d798ab140> was created in a different Context



📖 You are hiking just above 8,000ft when Sarah stops abruptly. She is breathless, pale, and leaning on her trekking poles. Her resting heart rate appears very high, and she has developed a dry, persistent cough. She complains of extreme fatigue and "tightness" in her chest. She is not asthmatic. How do you proceed?

Choices:
  1. Assume it is severe exhaustion and altitude fatigue. Encourage Sarah to rest for 30 minutes, hydrate, and then continue walking slowly.
  2. Suspect altitude-related pulmonary issue. Immediately initiate descent procedures and stop all further upward progress.


Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
2026/03/24 08:27:25 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f9db7adb4c0> at 0x7f9d798ace80> was created in a different Context
2026/03/24 08:27:36 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f9db7adb4c0> at 0x7f9d79788300> was created in a different Context
2026/03/24 08:27:36 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f9db7adb4c0> at 0x7f9d7933fac0> was created in a different Context
2026/03/24 08:27:36 WARNING mlflow.utils.autologging_utils: Encountered un

...

--- TURN 3 ---

📖 The mission concludes based on your decisions. In HAPE cases, immediate descent is the primary life-saving measure. Walking while symptomatic exacerbates pulmonary pressure and accelerates fluid accumulation in the lungs. Passive transport or extreme minimization of patient exertion is critical for a positive outcome. Rapid transport to an emergency room is required.

Choices:
  1. Scenario complete.
  2. Scenario complete.


Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
Deserializing unregistered type summit_sim.schemas.ChoiceOption from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ChoiceOption')]
Deserializing unregistered type summit_sim.schemas.SimulationResult from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'SimulationResult')]
2026/03/24 08:27:44 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f9db7adb4c0> at 0x7f9d790f5900> was created in a different Context
2026/03/24 08:27:54 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVa


=== SIMULATION COMPLETE ===

✓ Results logged to MLflow
✓ Total turns: 3
✓ Learning moments: 2
🏃 View run sim-demo-class-2024-hiking-3p-med at: https://mlflow.bhamm-lab.com/#/experiments/7/runs/0164f4d4b838410da7830cae1a04c051
🧪 View experiment at: https://mlflow.bhamm-lab.com/#/experiments/7


[Trace(trace_id=tr-7c131b6a7bcf9f8f5549684d672ee5ea), Trace(trace_id=tr-0fb7e8fa1db8561c0c9dfbeb61f0e58a)]

2026/03/24 08:28:00 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f9db7adb4c0> at 0x7f9d79241540> was created in a different Context


## 4. Phase 3: Debrief

View the comprehensive performance report generated automatically by the simulation graph.

In [25]:
debrief_report = current_state["debrief_report"]

print("=" * 60)
print("DEBRIEF REPORT")
print("=" * 60)

print(f"\n📊 Final Score: {debrief_report.final_score:.1f}%")
status_emoji = "✅" if debrief_report.completion_status == "pass" else "❌"
print(f"{status_emoji} Status: {debrief_report.completion_status.upper()}")

print("\n📝 Summary:")
print(debrief_report.summary)

if debrief_report.key_mistakes:
    print("\n⚠️ Key Mistakes:")
    for mistake in debrief_report.key_mistakes:
        print(f"  • {mistake}")

if debrief_report.strong_actions:
    print("\n💪 Strong Actions:")
    for action in debrief_report.strong_actions:
        print(f"  • {action}")

if debrief_report.teaching_points:
    print("\n📚 Teaching Points:")
    for point in debrief_report.teaching_points:
        print(f"  • {point}")

if debrief_report.best_next_actions:
    print("\n🎯 Recommendations:")
    for rec in debrief_report.best_next_actions:
        print(f"  • {rec}")

DEBRIEF REPORT

📊 Final Score: 66.7%
❌ Status: FAIL

📝 Summary:
The candidate demonstrated excellent clinical reasoning regarding the diagnosis and initial management of High Altitude Pulmonary Edema (HAPE). You correctly identified the critical nature of the condition and took appropriate, life-saving measures to descend and limit the patient's workload. However, the simulation was ended prematurely. In wilderness medical scenarios, the mission is not 'complete' simply by starting the descent; the provider is responsible for the patient until a formal hand-off to higher care is achieved. This oversight resulted in an incomplete management plan.

⚠️ Key Mistakes:
  • Premature termination of the scenario, indicating a lack of emphasis on continuous patient management during a medical emergency.
  • Failure to demonstrate readiness for the sustained effort required for a safe, managed evacuation.

💪 Strong Actions:
  • Correctly identified HAPE early in the presentation.
  • Prioritized

Trace(trace_id=tr-a1c23daf84aef2049fd34da518a4bc0f)

## 5. Results Summary

Complete transcript and learning moments from the simulation.

In [26]:
print("\n" + "=" * 60)
print("COMPLETE TRANSCRIPT")
print("=" * 60 + "\n")

for i, entry in enumerate(current_state["transcript"], 1):
    status = "✓" if entry["was_correct"] else "✗"
    print(f"Turn {i} {status}")
    print(f"  Situation: {entry['turn_narrative'][:80]}...")
    print(f"  Choice: {entry['choice_description']}")
    print(f"  Feedback: {entry['feedback'][:100]}...")
    print()


COMPLETE TRANSCRIPT

Turn 1 ✓
  Situation: You are hiking just above 8,000ft when Sarah stops abruptly. She is breathless, ...
  Choice: Suspect altitude-related pulmonary issue. Immediately initiate descent procedures and stop all further upward progress.
  Feedback: Excellent decision. By recognizing the potential for HAPE and initiating immediate descent, you have...

Turn 2 ✓
  Situation: You have correctly identified that staying at this altitude is dangerous. Sarah ...
  Choice: Prioritize organized, slow descent. Assist Sarah by carrying her pack, ensuring she minimizes personal exertion entirely.
  Feedback: Excellent call. By minimizing Sarah's exertion, you are directly reducing the strain on her heart an...

Turn 3 ✗
  Situation: The mission concludes based on your decisions. In HAPE cases, immediate descent ...
  Choice: Scenario complete.
  Feedback: While you reached the conclusion of the simulation, your final assessment path suggests an oversight...



In [27]:
print("\n" + "=" * 60)
print("KEY LEARNING MOMENTS")
print("=" * 60 + "\n")

for i, moment in enumerate(current_state["key_learning_moments"], 1):
    print(f"{i}. {moment}")


KEY LEARNING MOMENTS

1. Recognize that rapid ascent followed by respiratory distress (dry cough, tachypnea) and cyanosis is a clinical emergency, not just fatigue or mild altitude sickness.
2. The definitive field treatment for HAPE is immediate, passive descent; any continued movement or exertion by the patient further elevates pulmonary pressure and accelerates fluid accumulation in the lungs.


---
## ✅ Complete Workflow Demo

All phases executed:
- ✓ Scenario generated with AI
- ✓ Simulation completed with human-in-the-loop
- ✓ Debrief report generated
- ✓ All traces logged to MLflow

View traces in MLflow UI at your configured tracking URI.